# Fault-Tolerant Spark Structured Streaming

This notebook reads newline-delimited JSON files from a directory, tokenizes each message, and writes word records to Parquet. Structured Streaming uses a durable checkpoint to track source progress; the file source and Parquet file sink support recovery after a failure.

Keep the input, output, and checkpoint paths unchanged when restarting the query. Do not delete the checkpoint directory. Add data as new JSON files; do not modify files that have already been processed.

Each input line should contain an `event_time` timestamp and a `message` string. The setup creates sample JSON data automatically. Run this notebook in a Python environment with PySpark installed and Java configured.

In [ ]:
import json
import uuid
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, lower, regexp_replace, split
from pyspark.sql.types import StringType, StructField, StructType, TimestampType

# Keep these paths stable across query restarts.
base_dir = Path.cwd() / "spark_streaming_demo"
input_dir = base_dir / "input"
output_dir = base_dir / "output"
checkpoint_dir = base_dir / "checkpoint"
input_dir.mkdir(parents=True, exist_ok=True)

# Create reusable dummy JSON records on the first run.
sample_path = input_dir / "sample-events.json"
if not sample_path.exists():
    sample_events = [
        {"event_time": datetime.now(timezone.utc).isoformat(), "message": "Hello Spark streaming"},
        {"event_time": datetime.now(timezone.utc).isoformat(), "message": "Checkpointing supports fault tolerance"},
        {"event_time": datetime.now(timezone.utc).isoformat(), "message": "New JSON files are streamed automatically"},
    ]
    sample_path.write_text(
        "\n".join(json.dumps(event) for event in sample_events) + "\n",
        encoding="utf-8",
    )

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("FaultTolerantWordStream")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

schema = StructType([
    StructField("event_time", TimestampType(), nullable=False),
    StructField("message", StringType(), nullable=False),
])

events = (
    spark.readStream
    .schema(schema)
    .option("maxFilesPerTrigger", 1)
    .json(input_dir.as_uri())
)

words = (
    events
    .select(
        col("event_time"),
        explode(
            split(
                lower(regexp_replace(col("message"), r"[^a-zA-Z0-9 ]", " ")),
                r"\s+"
            )
        ).alias("word")
    )
    .where(col("word") != "")
)

# The persistent checkpoint tracks processed files; the Parquet sink is replay-safe.
query = (
    words.writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", output_dir.as_uri())
    .option("checkpointLocation", checkpoint_dir.as_uri())
    .start()
)

def add_event(message):
    """Write one new JSON event file for the active stream to discover."""
    event = {
        "event_time": datetime.now(timezone.utc).isoformat(),
        "message": message,
    }
    file_path = input_dir / f"events-{uuid.uuid4().hex}.json"
    file_path.write_text(json.dumps(event) + "\n", encoding="utf-8")
    return file_path

add_event("Spark streaming uses checkpointing for fault tolerance")
print("Streaming query started. Add more data with add_event(message).")

Run `add_event("another message")` in a new code cell to send more data. The stream runs asynchronously in the notebook. To inspect results, read the Parquet files under `spark_streaming_demo/output` after a micro-batch completes. To stop the query cleanly, run `query.stop()`. Restart using the same checkpoint directory to resume from saved progress.